# RMSE table

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

models_folder  = "*/Models"
results_folder = "*/Analysis"
os.makedirs(results_folder, exist_ok=True)

# model name: (folder, end of the file name)
models = {
    "Chronos2":    ("Chronos2", "chronos"),
    "ChronosBOLT": ("ChronosBOLT", "chronos_bolt"),
    "Moirai":      ("Moirai", "moirai"),
    "TimesFM":     ("TimesFM", "times_fm"),
}
context_lengths = [32, 64, 128, 256, 512]
walk_column = "walk_id"      # the column that says which walk a row belongs to

# read all prediction files into one table
tables = []
for model_name, (folder, file_end) in models.items():
    for L in context_lengths:
        file_path = f"{models_folder}/{folder}/predicitons_{L}_{file_end}.csv"
        predictions = pd.read_csv(file_path)
        predictions["model"] = model_name
        predictions["L"] = L
        tables.append(predictions)

data = pd.concat(tables, ignore_index=True)
data["snr"] = data["snr"].round(6)       # remove tiny float differences

# the forecasts start at the last context point, so the naive forecast is 0
print("naive is 0 everywhere:", np.allclose(data["naive"], 0))

# the plug-in forecast is horizon * context drift, so the context drift is plugin / horizon
data["context_drift"] = data["plugin"] / data["horizon"]

cell_columns = ["model", "L", "snr", "horizon"]    # one cell = all walks with the same settings
model_names = list(models)
print(data.shape)
data.head()

In [ ]:
# squared error against the realized value
data["se_model"]  = (data["forecast"]    - data["actual"]) ** 2
data["se_plugin"] = (data["plugin"]      - data["actual"]) ** 2
data["se_naive"]  = (data["naive"]       - data["actual"]) ** 2
data["se_cm"]     = (data["conditional"] - data["actual"]) ** 2     # noise floor

# RMSE = square root of the mean squared error over all walks in a cell
rmse_table = data.groupby(cell_columns)[["se_model", "se_plugin", "se_naive", "se_cm"]].mean().reset_index()
for name in ["model", "plugin", "naive", "cm"]:
    rmse_table[f"rmse_{name}"] = np.sqrt(rmse_table[f"se_{name}"])
    rmse_table = rmse_table.drop(columns=f"se_{name}")

rmse_table["model_vs_floor"] = rmse_table["rmse_model"] / rmse_table["rmse_cm"]    # 1 = noise floor

rmse_table.to_csv(f"{results_folder}/rmse_table.csv", index=False)
rmse_table

In [ ]:
# smaller table: only a few horizons, one column per horizon
chosen_horizons = [1, 5, 10, 20]

small_table = rmse_table[rmse_table["horizon"].isin(chosen_horizons)]
small_table = small_table.pivot(index=["model", "L", "snr"], columns="horizon", values="rmse_model").round(3)

small_table.to_csv(f"{results_folder}/rmse_table_small.csv")
small_table